## Importing libraries

In [1]:
import numpy as np
import pandas as pd

## Importing data

Columns to drop

In [2]:
columns_to_not_use = [
# Same values for all rows
    'record_type' ,
    'record_subclass' ,
    'confirmation_location' ,
    'dia_bp' ,
    'group_number' ,
    'manufacturer_code' ,
    'patient_age_units' ,
    'patient_height' ,
    'patient_weight' ,
    'reason_for_procedure' ,
    'sys_bp' ,
    'text_size' ,
    'pp_interval' ,

# No value at all
    'assisting_provider' ,
    'attending_phy' ,
    'patient_middle_name' ,
    'patient_secondary_id' ,
    'patient_ssn' ,
    'procedural_diagnosis' ,
    'referral_source' ,
    'referring_phy' ,
]

Columns to format in a particular type

In [ ]:
sure_string_cols = [
    'pseudopaz'
]
dtype = {col: 'string' for col in sure_string_cols}

string_cat_cols = [
    'site_number', # ???
    'record_status',
    'medications',
    'patient_height_units' ,
    'patient_location' ,
    'patient_race' ,
    'patient_race_str' ,
    'patient_room' ,
    'patient_sex' ,
    'patient_weight_units' ,
    'transmit_device_number' ,
    'reason_for_change' ,
]
dtype.update({col: 'string' for col in string_cat_cols})

Import the excel file

In [4]:
df = pd.read_excel('../raw_data/dati_escribe.xlsx',
    usecols=lambda col: col not in columns_to_not_use,
    dtype_backend='numpy_nullable', # For allowing the `string` dtype
    dtype=dtype
)

# Make the index start from 2 (as the original excel)
df.index = df.index + 2

## Parsing and reformatting

### Merging similar values in columns

#### `comment_field`

Values are `{..., 'Dolore toracico': 8,'Dolore torace': 2, ...}` - `Dolore torace` values are merged to `Dolore toracico`

In [5]:
df['comment_field'] = df['comment_field'].replace('Dolore torace', 'Dolore toracico')

#### `note1`

Merging:
- `DOL. TORACICO` and `DOL TOR` and `DOLORETORAC` and `DOLORE TORACICO`
    - to `TORACALGIA`
- `ADDOMINALG` and `DOLORE ADDOMIN`
    - to `ADDOMINALGIA`
- `Dispnea`
    - to `DISPNEA`

In [6]:
df['note1'] = df['note1'].replace(['DOLORE TORACICO', 'DOLORETORAC', 'DOL. TORACICO', 'DOL TOR'], 'TORACALGIA')
df['note1'] = df['note1'].replace(['ADDOMINALG', 'DOLORE ADDOMIN'], 'ADDOMINALGIA')
df['note1'] = df['note1'].replace(['Dispnea'], 'DISPNEA')

#### `note2` (tanti auguri)

Merging:
- `.` and `.`
    - to `.`

In [7]:
# df[''] = df[''].replace([''], '')

#### `patient_location`

Merging:
- `OTO`
    - to `Oto`

In [8]:
df['patient_location'] = df['patient_location'].replace(['OTO'], 'Oto')

#### ``

Merging:
- `.` and `.`
    - to `.`

In [9]:
# df[''] = df[''].replace([''], '')

#### ``

Merging:
- `.` and `.`
    - to `.`

In [10]:
# df[''] = df[''].replace([''], '')

### Splitting `interp_text` column

Original column can keeped decommenting line ***

In [ ]:
def split_column(value):
    """
    Requires pandas.
    """
    # Check if value is NA/NaN/None
    if pd.isna(value):
        return pd.Series([pd.NA, pd.NA])

    # Split on '~~' to get col1
    parts = value.split('~~', 1)
    
    return pd.Series([parts[0], parts[1]])


# Get position and split
original_col_name = 'interp_text'
original_position = df.columns.get_loc(original_col_name)

# Split the column
split_df = df[original_col_name].apply(split_column)
split_df.columns = ['AIC_name', 'AIC_form', 'AIC_qty']
split_df = split_df.astype('string')

# Reconstruct dataframe: columns before + new columns + columns after
df = pd.concat([
    df.iloc[:, :original_position
        # +1 # Decomment for keeping the original column
    ],
    split_df,
    df.iloc[:, original_position+1:]
], axis=1)

In [ ]:
# def split_column(value):
#     """
#     Requires pandas.
#     """
#     # Check if value is NA/NaN/None
#     if pd.isna(value):
#         return pd.Series([pd.NA, pd.NA, pd.NA])

#     # Split on '~~' to get col1
#     parts = value.split('~~', 1)
#     if len(parts) != 2:
#         return pd.Series([pd.NA, pd.NA, pd.NA])
    
#     col1 = parts[0]
#     remainder = parts[1]
    
#     # Find the last space followed by a number (handles comma as decimal separator)
#     match = re.search(r'^(.+)\s+(\d+(?:,\d+)?.*?)$', remainder)
    
#     if match:
#         col2 = match.group(1)
#         col3 = match.group(2)
#     else:
#         col2 = remainder
#         col3 = None
    
#     return pd.Series([col1, col2, col3])


# # Get position and split
# original_col_name = 'AIC_DES'
# original_position = df.columns.get_loc(original_col_name)

# # Split the column
# split_df = df[original_col_name].apply(split_column)
# split_df.columns = ['AIC_name', 'AIC_form', 'AIC_qty']
# split_df = split_df.astype('string')

# # Reconstruct dataframe: columns before + new columns + columns after
# df = pd.concat([
#     df.iloc[:, :original_position
#         # +1 # Decomment for keeping the original column
#     ],
#     split_df,
#     df.iloc[:, original_position+1:]
# ], axis=1)

### Formatting `patient_date_of_birth` column

Since 33 rows have the date formatted as `06/17/1950` (for example row 790), `format='mixed'` is required

In [11]:
df['patient_date_of_birth'] = pd.to_datetime(df['patient_date_of_birth'], errors='raise', dayfirst=True, format='mixed')

### Formatting `overreading_phy` column

In [12]:
def format_phy_name(value):
    """
    Requires pandas.
    TODO
    """
    # Handle NA values
    if pd.isna(value):
        return pd.NA

    # Count whitespaces
    dot_count = value.count('.')

    # Handle the only name with one dot
    if dot_count == 1:
        return value

    elif dot_count == 2:
        second_dot = value.rfind('.')
        name = value[second_dot -1 : second_dot]
        surname = value[value.rfind('.') +1 : ].strip()
        formatted_name = 'DR-' + name + '_' + surname
        
        return formatted_name

# Only name with one dot
df['overreading_phy'] = df['overreading_phy'].replace('DR. GP BROSO', 'DR-GP_BROSO')

# Run the function
df['overreading_phy'] = df['overreading_phy'].apply(format_phy_name).astype('string')

### Inconsistency between `pseudopaz` and other columns

#### Proof of the inconsistency

`patient_age`, `patient_date_of_birth`, `patient_race`, `patient_sex` are inconsistent

In [13]:
# (df.groupby('pseudopaz')['patient_age'].nunique() == 1).all()
# (df.groupby('pseudopaz')['patient_date_of_birth'].nunique() == 1).all()
# (df.groupby('pseudopaz')['patient_race'].nunique() == 1).all()
# (df.groupby('pseudopaz')['patient_sex'].nunique() == 1).all()

#### See which `pseudopaz` have multiple characteristics that should be unique

In [14]:
# incons = df.groupby('pseudopaz')[''].nunique()
# incons = incons[incons > 1].sort_values(ascending=False)
# incons

## Final checks

In [15]:
df['transmit_device_number']

2       0
3       0
4       0
5       0
6       0
       ..
1784    4
1785    0
1786    0
1787    0
1788    0
Name: transmit_device_number, Length: 1787, dtype: string

In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1787 entries, 2 to 1788
Data columns (total 39 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   pseudopaz               1787 non-null   string        
 1   site_number             1787 non-null   string        
 2   record_status           1787 non-null   string        
 3   acq_time                1787 non-null   datetime64[us]
 4   confirm_time            886 non-null    datetime64[us]
 5   transmission_time       1787 non-null   datetime64[us]
 6   comment_field           14 non-null     string        
 7   medications             1787 non-null   string        
 8   note1                   144 non-null    string        
 9   note2                   1047 non-null   string        
 10  overreading_phy         873 non-null    string        
 11  patient_age             1787 non-null   Int64         
 12  patient_date_of_birth   1787 non-null   datetime64[us]
 13 

## Export

In [17]:
df.to_parquet("../raw_data/parquets/dati_escribe.parquet"
    # , compression="gzip"
)